# Exercise 01: GANs and VAEs
## AIAT 122 - Deep Learning | Unit 4

## Learning Objectives

- Build a simple **Generator** and **Discriminator** (GAN) or **Encoder/Decoder** (VAE)
- Train on image data (e.g. MNIST) for generation or reconstruction
- Visualize generated or reconstructed samples
- Relate to real-world use: image generation, anomaly detection, compression

## Real-World Context

GANs are used for **image generation**, **style transfer**, and **data augmentation**. VAEs are used for **anomaly detection**, **compression**, and **latent representation**. This exercise aligns with slides on AutoEncoders (04), GANs (09), and VAEs (22).

**Task:** Implement either a **simple GAN** (generator + discriminator on MNIST) or a **simple VAE** (encoder + decoder with latent).

**Theory → Practice:** This exercise applies the theory and code patterns from `01_gans_and_autoencoders_vaes.ipynb` and `02_implementing_a_vae_variational_autoencoder_for_anomaly_detection.ipynb`. Complete those examples first if you haven't.

---

## Task 1: Model Architecture (35 points)

**Option A – GAN:**
1. Define a **Generator** that maps noise (e.g. 100-dim) to a 28×28 image (MNIST).
2. Define a **Discriminator** that classifies real vs fake images (binary).

**Option B – VAE:**
1. Define an **Encoder** (input → mean and log_variance of latent).
2. Define a **Decoder** (latent sample → reconstruction).
3. Implement reparameterization: sample = mean + std * epsilon.

Use concepts from the unit examples (`01_gans_and_autoencoders_vaes.ipynb`, `02_implementing_a_vae_variational_autoencoder_for_anomaly_detection.ipynb`).

## 📥 Inputs & 📤 Outputs

**Inputs:** MNIST (or subset), PyTorch/TensorFlow, concepts from Unit 4 examples.

**Outputs:** Trained generator/decoder, loss curves, and a figure showing **generated** (GAN) or **original vs reconstructed** (VAE) images with axis labels.

**Expected:** When complete, you should see loss curves (e.g. G vs D loss or VAE reconstruction + KL) and a figure of generated or reconstructed images. Compare with the instructor solution after the deadline.

In [1]:
%pip install torch torchvision matplotlib -q
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# TODO: Load MNIST (or subset), normalize, create DataLoader
# YOUR CODE HERE
print('Data loaded.')

Note: you may need to restart the kernel to use updated packages.


Data loaded.


### Task 1 (continued): Define Generator + Discriminator (GAN) or Encoder + Decoder (VAE)

In [2]:
# TODO: Implement Generator and Discriminator (GAN) OR Encoder and Decoder (VAE)
# GAN: Generator(noise_dim=100) -> (batch, 1, 28, 28); Discriminator(image) -> (batch, 1)
# VAE: Encoder(x) -> mean, log_var; Decoder(z) -> reconstruction
class Generator(nn.Module):
    # YOUR CODE HERE
    pass

class Discriminator(nn.Module):
    # YOUR CODE HERE
    pass

# Or for VAE: class Encoder, class Decoder, class VAE

## Task 2: Training Loop (40 points)

**GAN:** Alternate training: (1) Train discriminator on real + fake batches, (2) Train generator to fool discriminator.

**VAE:** Minimize reconstruction loss + KL divergence (or simple MSE + latent regularization).

Train for a few epochs (e.g. 5–10) so the run completes in a reasonable time. Store loss(es) for plotting.

In [3]:
# TODO: Implement training loop (GAN or VAE)
# - GAN: optimizer_d, optimizer_g; update D then G each step
# - VAE: single optimizer; loss = reconstruction + beta * KL
# YOUR CODE HERE
print('Training complete.')

Training complete.


## Task 3: Visualize and Evaluate (25 points)

1. Plot **loss curve(s)** (e.g. discriminator/generator loss or VAE total loss) with **xlabel**, **ylabel**, and **title**.
2. **GAN:** Show a grid of **generated** images (e.g. 4×4) from random noise.
3. **VAE:** Show **original vs reconstructed** images (e.g. 5 pairs) with clear labels.

**Submission:** Complete notebook with model definitions, training, and figures.

**Grading:** 100 points total (35 + 40 + 25).

In [4]:
import matplotlib.pyplot as plt

# TODO: Plot loss(es) with plt.xlabel, plt.ylabel, plt.title
# TODO: Visualize generated (GAN) or original vs reconstructed (VAE) images
# YOUR CODE HERE

## 🌍 Real-World Worked Example — GAN Generating Handwritten Digits

**Industry context:**
- NVIDIA uses GANs to generate synthetic training data for autonomous vehicles
- Pharmaceutical companies use GANs to generate molecular structures for drug discovery
- Fashion brands (Zalando, H&M) use GANs to generate clothing designs

We train a **DCGAN** to generate realistic handwritten digit images from pure noise.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
LATENT_DIM = 64; BATCH = 128

transform = T.Compose([T.ToTensor(), T.Normalize([0.5],[0.5])])
dataset   = torchvision.datasets.MNIST('/tmp/mnist', train=True, download=True, transform=transform)
loader    = torch.utils.data.DataLoader(dataset, batch_size=BATCH, shuffle=True)

# ── Generator: noise → image ───────────────────────────────────────────────
G = nn.Sequential(
    nn.Linear(LATENT_DIM, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 512),        nn.LeakyReLU(0.2),
    nn.Linear(512, 28*28),      nn.Tanh()
)
# ── Discriminator: image → real/fake ─────────────────────────────────────
D = nn.Sequential(
    nn.Linear(28*28, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(512, 256),   nn.LeakyReLU(0.2), nn.Dropout(0.3),
    nn.Linear(256, 1),     nn.Sigmoid()
)
opt_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
opt_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
bce   = nn.BCELoss()

d_losses, g_losses = [], []
for epoch in range(10):
    for real_imgs, _ in loader:
        bs = real_imgs.size(0)
        real_flat = real_imgs.view(bs, -1)
        # ── Train Discriminator ─────────────────────────────────────────
        z    = torch.randn(bs, LATENT_DIM)
        fake = G(z).detach()
        loss_D = bce(D(real_flat), torch.ones(bs,1)) + bce(D(fake), torch.zeros(bs,1))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
        # ── Train Generator ─────────────────────────────────────────────
        z      = torch.randn(bs, LATENT_DIM)
        fake   = G(z)
        loss_G = bce(D(fake), torch.ones(bs,1))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    d_losses.append(loss_D.item()); g_losses.append(loss_G.item())
    print(f"Epoch {epoch+1}/10 — D loss: {d_losses[-1]:.3f}  G loss: {g_losses[-1]:.3f}")

# ── Show generated images ──────────────────────────────────────────────────
G.eval()
with torch.no_grad():
    samples = G(torch.randn(16, LATENT_DIM)).view(16, 28, 28).numpy()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i], cmap='gray'); ax.axis('off')
plt.suptitle("GAN-Generated Digits (from pure noise) — Same tech as DALL-E and Midjourney")
plt.tight_layout(); plt.show()

## 📝 Summary

In this notebook, you practiced:
- Building generative architectures: either a **GAN** (Generator + Discriminator) or a **VAE** (Encoder + Decoder with reparameterization)
- Training on image data (MNIST) to produce generated or reconstructed samples
- Understanding adversarial loss (GAN) vs reconstruction + KL-divergence loss (VAE)
- Visualizing generated/reconstructed images to assess model quality

**Next steps:** Explore conditional GANs (cGANs) or hierarchical VAEs to control the style of generated outputs — building toward the text/image generation models in Course 10.

## 📚 References & Further Reading

**Foundational Papers:**
- Goodfellow et al. (2014) — [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661) *(invented GANs)*
- Radford et al. (2015) — [DCGAN](https://arxiv.org/abs/1511.06434)
- Karras et al. (2020) — [StyleGAN2](https://arxiv.org/abs/1912.04958)

**State-of-the-Art:**
- Midjourney and DALL-E 2 build on GAN + diffusion ideas
- Deepfake detection (Meta, Microsoft) uses GAN discriminators as detectors